# Utilities to modify the eigenvalues

In [ ]:
# | default_exp utils.eigenvalues

In [ ]:
# | exporti

import jax
import jax.numpy as jnp
from jax.typing import ArrayLike

In [ ]:
# | export


def multiply_eigenvalues(
    eigenvalues: ArrayLike,
    angle_factor: float,
    radius_factor: float,
    minimum_radius: float = 0.01,  # Minimum radius to avoid zero eigenvalues
) -> ArrayLike:
    """
    Modify the angles of eigenvalues within the unit circle based on the factor.

    Args:
        eigenvalues (jnp.ndarray): Complex eigenvalues within the unit circle.
        factor (float): The multiplication factor for the angles.

    Returns:
        jnp.ndarray: Modified eigenvalues.
    """
    # Decompose eigenvalues into radius and angle
    radii = jnp.abs(eigenvalues)
    angles = jnp.angle(eigenvalues)

    # Determine the adjustment for eigenvalues
    upper_half = angles >= 0  # Eigenvalues in the upper half-plane
    # lower_half = ~upper_half  # Eigenvalues in the lower half-plane

    # Adjust angles
    adjusted_angles = jnp.where(
        upper_half,
        angles * angle_factor,  # Rotate clockwise
        angles * angle_factor,  # Rotate anti-clockwise
    )

    # Adjust radii
    radii = radii * radius_factor

    # Check Nyquist condition (|frequency| > Nyquist -> radius = 0)
    nyquist_limit = jnp.pi
    exceeded_nyquist = jnp.abs(adjusted_angles) > nyquist_limit
    radii = jnp.where(exceeded_nyquist, minimum_radius, radii)

    # Reconstruct eigenvalues
    new_eigenvalues = radii * jnp.exp(1j * adjusted_angles)
    return new_eigenvalues

In [ ]:
# | export


def ensure_positive_imaginary_parts(eigenvalues: ArrayLike) -> ArrayLike:
    # Ensure all reconstructed eigenvalues have positive imaginary parts
    eigenvalues = jnp.where(
        eigenvalues.imag < 0,
        eigenvalues + 2j * jnp.pi,
        eigenvalues,
    )
    return eigenvalues

In [ ]:
# | test

# create eigenvalues in the laplace domain
key = jax.random.PRNGKey(0)
theta = jax.random.uniform(key, (40,), minval=0, maxval=jnp.pi)
nu = jax.random.uniform(key, (40,), minval=0, maxval=0.02)

eigenvalues = -nu + 1j * theta
discrete_eigenvalues = jnp.exp(eigenvalues)

discrete_eigenvalues_mod = multiply_eigenvalues(discrete_eigenvalues, 1.0, 1.0)

# Convert back to continuous eigenvalues
reconstructed_eigenvalues = jnp.log(discrete_eigenvalues_mod)

adjusted_reconstructed_eigenvalues = ensure_positive_imaginary_parts(
    reconstructed_eigenvalues
)

# Check equivalence
assert jnp.allclose(eigenvalues, adjusted_reconstructed_eigenvalues, atol=1e-6), (
    "Eigenvalues reconstruction failed!"
)

In [ ]:
# | export


def filter_eigenvalues(
    eigenvalues: ArrayLike,
    angle_min_threshold,
    angle_max_threshold,
    magnitude_min_threshold,
    magnitude_max_threshold,
    eps=1e-5,
) -> ArrayLike:
    """
    Filter eigenvalues based on angle and magnitude thresholds.

    Args:
        eigenvalues (ArrayLike): Eigenvalues to filter.
        angle_min_threshold (float): Minimum angle threshold.
        angle_max_threshold (float): Maximum angle threshold.
        magnitude_min_threshold (float): Minimum magnitude threshold.
        magnitude_max_threshold (float): Maximum magnitude threshold.

    Returns:
        jnp.ndarray: Filtered eigenvalues.
    """
    # Decompose eigenvalues into radius and angle
    radii = jnp.abs(eigenvalues)
    angles = jnp.angle(eigenvalues)

    # Filter eigenvalues based on angle and magnitude thresholds
    filtered_eigenvalues = jnp.where(
        (angles >= angle_min_threshold)
        & (angles <= angle_max_threshold)
        & (radii >= magnitude_min_threshold)
        & (radii <= magnitude_max_threshold),
        eigenvalues,
        eps + 0.0j,
    )
    return filtered_eigenvalues

In [ ]:
# | test

# create random eigenvalues in the unit circle
key = jax.random.PRNGKey(0)
angles = jax.random.uniform(key, (256,), minval=0, maxval=jnp.pi * 2)
radii = jnp.sqrt(jax.random.uniform(key + 1, (256,), minval=0.01, maxval=1))

eigenvalues = radii * jnp.exp(1j * angles)
filtered_eigenvalues = filter_eigenvalues(eigenvalues, 0.0, jnp.pi, 0.2, 0.8)

assert jnp.all(jnp.angle(filtered_eigenvalues) >= 0.0)
assert jnp.all(jnp.angle(filtered_eigenvalues) <= jnp.pi)
assert jnp.all(jnp.abs(filtered_eigenvalues) <= 0.8)